# Building Conversational RAG with OpenAI and Chroma DB: No LangChain or LlamaIndex Required!

## Table of Contents
1. Document Processing and Indexing
2. Setting Up ChromaDB
3. Inserting Data into ChromaDB
4. Semantic Search on ChromaDB
5. Combining ChromaDB and OpenAI for RAG
6. Creating Conversational RAG with Memory


###What is Retrieval Augmented Generation (RAG)?

RAG is a technique that enhances language models by combining them with a retrieval system. It allows the model to access and utilize external knowledge when generating responses.

## Installing Necessary Libraries

In [ ]:
#!pip install -qU chromadb pypdf2 python-docx sentence-transformers

In [ ]:
!pip install -qU chromadb pypdf2 python-docx sentence-transformers

In [ ]:
!pip show chromadb pypdf2 python-docx sentence-transformers

In [ ]:
!pip install -qU google-genai openai

In [ ]:
#!pip install -qU google-genai openai
from google import genai

## Document Processing and Indexing

###Functions to read file contents

In [ ]:
#import docx
import PyPDF2
import os
def read_text_file(file_path: str):
    """Read content from a txt file"""
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def read_pdf_file(file_path: str):
    """Read content from a PDF file"""
    text = ""
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            text += page.extract_text() + "\n"
    return text

# def read_docx_file(file_path: str):
#     """Read content from a docx file"""
#     doc = docx.Document(file_path)
#     return "\n".join([paragraph.text for paragraph in doc.paragraphs])

In [ ]:
def read_document(file_path: str):
    """Read document content based on file extension"""
    _, file_extension = os.path.splitext(file_path)
    file_extension = file_extension.lower()

    if file_extension == '.txt':
        return read_text_file(file_path)
    elif file_extension == '.pdf':
        return read_pdf_file(file_path)
    elif file_extension == '.docx':
        return read_docx_file(file_path)
    else:
        raise ValueError(f"Unsupported file format: {file_extension}")

text = read_document(r"A:\DEPLOYMENT_RAG\ERP-2008-chapter4.pdf")
#text = read_document(r"/content/ERP-2008-chapter4.pdf") usinig this in the colab
print(text)

###Chunking

In [ ]:
def split_text(text: str, chunk_size: int = 500):
    """Split text into chunks"""
    sentences = text.replace('\n', ' ').split('. ')
    chunks = []
    current_chunk = []
    current_size = 0
    print(f"no of sentences:  {len(sentences)}")
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        if not sentence.endswith('.'):
            sentence += '.'

        sentence_size = len(sentence)

        if current_size + sentence_size > chunk_size and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentence]
            current_size = sentence_size
        else:
            current_chunk.append(sentence)
            current_size += sentence_size

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

chunks = split_text(text)
print(chunks[2])

In [ ]:
text = "Hi hello how are"

len(text)

In [ ]:
len(chunks)


## Setting Up ChromaDB

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
# import textwrap

In [ ]:
client = chromadb.PersistentClient(path="./chroma_db")

# Use sentence-transformer embeddings for embedding our data
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(name="documents_collection", embedding_function=sentence_transformer_ef)

In [ ]:
# from sentence_transformers import SentenceTransformer
# sentences = ["This is an example sentence", "Each sentence is converted"]

# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# embeddings = model.encode(sentences)
# print(embeddings)

## Inserting Data into ChromaDB

In [ ]:
def process_document(file_path: str):
    """Process a single document and prepare it for ChromaDB"""
    try:
        # Read the document
        content = read_document(file_path)

        # Split into chunks
        chunks = split_text(content)

        # Prepare metadata
        file_name = os.path.basename(file_path)
        metadatas = [{"source": file_name, "chunk": i} for i in range(len(chunks))]
        ids = [f"{file_name}_chunk_{i}" for i in range(len(chunks))]

        return ids, chunks, metadatas
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")
        return [], [], []

def add_to_collection(collection, ids, texts, metadatas):
    """Add documents to collection in batches"""
    if not texts:
        return

    batch_size = 100
    for i in range(0, len(texts), batch_size):
        end_idx = min(i + batch_size, len(texts))
        collection.add(
            documents=texts[i:end_idx],
            metadatas=metadatas[i:end_idx],
            ids=ids[i:end_idx]
        )

def process_and_add_documents(collection, folder_path: str):
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Folder path not found: {folder_path}")

    supported_extensions = {".txt", ".pdf", ".docx"}
    files = [
        os.path.join(folder_path, file)
        for file in os.listdir(folder_path)
        if os.path.isfile(os.path.join(folder_path, file))
        and os.path.splitext(file)[1].lower() in supported_extensions
    ]

    if not files:
        print(f"No supported document files found in {folder_path}")
        return

    for file_path in files:
        print(f"Processing {os.path.basename(file_path)}...")
        ids, texts, metadatas = process_document(file_path)
        add_to_collection(collection, ids, texts, metadatas)
        print(f"Added {len(texts)} chunks to collection")

In [ ]:
process_and_add_documents(collection, r"A:\\DEPLOYMENT_RAG")

## Semantic Search on ChromaDB

In [ ]:
def semantic_search(collection, query: str, n_results: int = 4):
    """Perform semantic search on the collection"""
    return collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["embeddings", "documents", "metadatas", "distances"]
    )

In [ ]:
query = "What is American Health System?"
results = semantic_search(collection, query)
results

# Chroma’s response structure
#     ids
#     embeddings
#     documents
#     metadatas
#     distances
#     uris
#     data
#     included

In [ ]:
def print_search_results(results):
    """Print formatted search results"""
    print("\nSearch Results:\n" + "-" * 50)

    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        meta = results['metadatas'][0][i]
        print(f"\nResult {i + 1}: Source: {meta['source']}, Chunk {meta['chunk']}")
        print(f"Content: {doc}\n")

print_search_results(results)

In [ ]:
def get_context_with_sources(results):
    """Get a combined context and formatted sources from search results."""
    # Combine the document chunks into a single context
    context = "\n\n".join(results['documents'][0])

    # Format the sources with metadata information
    sources = [f"{meta['source']} (chunk {meta['chunk']})" for meta in results['metadatas'][0]]

    return context, sources

context, sources = get_context_with_sources(results)
print(context)

## Combining ChromaDB and Gemini for RAG

In [ ]:
def get_prompt(query: str, context: str):
    """Prompt for Response Generation"""
    prompt = f"""Based on the following context, please answer the question.
    If the answer cannot be derived from the context, say "I cannot answer this based on the provided context."

    Context:
    {context}

    Question: {query}

    Answer:"""

    return prompt

In [ ]:
import os
from dotenv import load_dotenv

try:
    from openai import OpenAI
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "openai"])
    from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [ ]:
def generate_response_openai_style(query: str, context: str):
    """Generate a response using OpenAI"""

    prompt = get_prompt(query, context)
    print(prompt)
    
    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers questions based on the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=500
    )

    return response.choices[0].message.content

In [ ]:
def generate_response_gemini_new(query: str, context: str):
    """Generate a response using OpenAI"""

    prompt = get_prompt(query, context)
    #print(prompt)
    
    client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)
    
    response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
    )

    return response.text

In [ ]:
def rag_query(collection, query: str, n_chunks: int = 2):
    """Perform RAG query: retrieve relevant chunks and generate answer"""
    # Get relevant chunks
    results = semantic_search(collection, query, n_chunks)
    context, sources = get_context_with_sources(results)

    # Generate response
    response = generate_response_openai_style(query, context)
    response = generate_response_gemini_new(query, context)

    return response, sources

Functions to get prompt and OpenAI Response

##Perform RAG query

In [ ]:
query = "What is the demand for health"
response, sources = rag_query(collection, query)

# Print results
#print("\nQuery:", query)
print("\nAnswer:", response)
#print("\nSources used:")
# for source in sources:
#     print(f"- {source}")